# Experiment 01 — HUGS Locality Probe (Colab v2)

This notebook reproduces the HUGS baseline and runs the first locality diagnostic for **Interactive Digital Humans / 4D Human Intelligence**.

**Question:** when a local SMPL joint is perturbed, how much of the Gaussian human representation changes outside the intended semantic region?

Use an **A100 GPU** if available. This version avoids shell heredocs and creates an isolated Python 3.8 / PyTorch 1.13.1 / CUDA 11.7 environment inside Colab.


## 0. Verify the Colab GPU


In [ ]:
!nvidia-smi
import torch, platform
print('Colab Python:', platform.python_version())
print('Colab PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Enable a GPU runtime before continuing.'


## 1. Clone the research repo and HUGS


In [ ]:
import subprocess, os, shutil
from pathlib import Path

def run(cmd, cwd=None, env=None):
    print('+', cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd, env=env, executable='/bin/bash')

run('rm -rf /content/interactive-digital-humans /content/ml-hugs')
run('git clone https://github.com/reusahn/interactive-digital-humans.git /content/interactive-digital-humans')
run('git clone --recursive https://github.com/apple-aiml-research/ml-hugs.git /content/ml-hugs')


## 2. Build a pinned HUGS environment

HUGS was released against Python 3.8, PyTorch 1.13.1 and CUDA 11.7. Current Colab uses a much newer stack, so we keep the Colab kernel untouched and create a separate Miniforge environment. We also install a CUDA 11.7 development toolkit because HUGS compiles CUDA extensions.

This cell can take several minutes. If it fails, send the **first error block**, not only the final line.


In [ ]:
from pathlib import Path
import os, subprocess

MINIFORGE = Path('/content/miniforge')
CONDA = MINIFORGE / 'bin/conda'
ENV = MINIFORGE / 'envs/hugs'

if not CONDA.exists():
    run('wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /content/miniforge.sh')
    run('bash /content/miniforge.sh -b -p /content/miniforge')

run(f'{CONDA} env remove -n hugs -y || true')
run(f'{CONDA} create -n hugs python=3.8 pip -y -c conda-forge')

# Full CUDA 11.7 development toolkit supplies nvcc for the HUGS CUDA extensions.
run(f'{CONDA} install -n hugs -y -c nvidia/label/cuda-11.7.0 cuda-toolkit=11.7.0')

PIP = ENV / 'bin/pip'
PYTHON = ENV / 'bin/python'
run(f"{PIP} install --upgrade 'pip<25' 'setuptools<70' wheel ninja")
run(f"{PIP} install torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117")
run(f'{PIP} install numpy==1.23.5 fvcore iopath')
run(f'{PIP} install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/py38_cu117_pyt1131/download.html')

build_env = os.environ.copy()
build_env['CUDA_HOME'] = str(ENV)
build_env['PATH'] = f"{ENV}/bin:" + build_env['PATH']
build_env['LD_LIBRARY_PATH'] = f"{ENV}/lib:" + build_env.get('LD_LIBRARY_PATH', '')
build_env['TORCH_CUDA_ARCH_LIST'] = '8.0'  # A100

run(f'{PIP} install submodules/diff-gaussian-rasterization', cwd='/content/ml-hugs', env=build_env)
run(f'{PIP} install submodules/simple-knn', cwd='/content/ml-hugs', env=build_env)
run(f'{PIP} install -r requirements.txt', cwd='/content/ml-hugs', env=build_env)
run(f'{PIP} install git+https://github.com/mattloper/chumpy.git', cwd='/content/ml-hugs', env=build_env)

assert ENV.exists(), f'HUGS environment was not created: {ENV}'
print('HUGS env created at:', ENV)


## 3. Verify the isolated HUGS environment


In [ ]:
verify = [
    str(PYTHON), '-c',
    "import torch; print('HUGS torch:', torch.__version__); print('CUDA runtime:', torch.version.cuda); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')"
]
subprocess.run(verify, check=True)
nvcc = ENV / 'bin/nvcc'
print('nvcc exists:', nvcc.exists())
if nvcc.exists():
    subprocess.run([str(nvcc), '--version'], check=True)


## 4. Download NeuMan data and HUGS pretrained models


In [ ]:
run('bash scripts/prepare_data_models.sh', cwd='/content/ml-hugs')
print('Downloaded HUGS data/models.')


## 5. Upload the licensed SMPL neutral model

Download the SMPL neutral body model v1.1.0 from the official SMPL site. Upload the `.pkl` file below. It will be renamed to `SMPL_NEUTRAL.pkl`. `smpl_uv.obj` is optional for this probe.


In [ ]:
from google.colab import files
uploaded = files.upload()
smpl_dir = Path('/content/ml-hugs/data/smpl')
smpl_dir.mkdir(parents=True, exist_ok=True)
for name in uploaded:
    src = Path(name)
    if name.lower().endswith('.pkl'):
        dst = smpl_dir / 'SMPL_NEUTRAL.pkl'
    elif name == 'smpl_uv.obj':
        dst = smpl_dir / 'smpl_uv.obj'
    else:
        continue
    shutil.move(str(src), str(dst))
    print('Saved:', dst)
assert (smpl_dir / 'SMPL_NEUTRAL.pkl').exists(), 'Upload the licensed SMPL neutral .pkl file.'


## 6. Automatically choose a pretrained HUGS experiment


In [ ]:
root = Path('/content/ml-hugs')
candidates = []
for cfg in root.rglob('config_train.yaml'):
    d = cfg.parent
    human_ckpts = list(d.glob('*human*.pth')) + list((d / 'ckpt').glob('*human*.pth')) if (d / 'ckpt').exists() else list(d.glob('*human*.pth'))
    if human_ckpts:
        candidates.append(d)
print('Pretrained human candidates:', len(candidates))
for i, d in enumerate(candidates):
    print(i, d)
assert candidates, 'No pretrained HUGS human experiment was found after extraction.'
preferred = [d for d in candidates if 'lab' in str(d).lower()]
HUGS_OUTPUT_DIR = str(preferred[0] if preferred else candidates[0])
print('Selected:', HUGS_OUTPUT_DIR)


## 7. Sanity-check the official HUGS evaluation


In [ ]:
eval_env = build_env.copy()
subprocess.run([str(PYTHON), 'scripts/evaluate.py', '-o', HUGS_OUTPUT_DIR], cwd='/content/ml-hugs', env=eval_env, check=True)


## 8. Run Experiment 01: left-wrist locality probe

The first probe adds +10 degrees around the left wrist z-axis and measures Gaussian displacement inside vs outside the semantic target region. One run is a smoke test, not a scientific conclusion.


In [ ]:
probe_dir = '/content/interactive-digital-humans/experiments/01-baseline'
probe_cmd = [
    str(PYTHON), 'hugs_locality_probe.py',
    '--hugs-root', '/content/ml-hugs',
    '--output-dir', HUGS_OUTPUT_DIR,
    '--frame', '0',
    '--joint', 'left_wrist',
    '--axis', 'z',
    '--degrees', '10',
    '--save-dir', '/content/probe_results',
]
subprocess.run(probe_cmd, cwd=probe_dir, env=eval_env, check=True)


In [ ]:
import json, glob
result_files = sorted(glob.glob('/content/probe_results/frame*.json'))
assert result_files, 'No probe result JSON was produced.'
with open(result_files[-1]) as f:
    result = json.load(f)
result


## 9. Save outputs to Google Drive


In [ ]:
from google.colab import drive
import datetime
drive.mount('/content/drive')
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = Path('/content/drive/MyDrive/interactive-digital-humans/experiment-01') / stamp
dst.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree('/content/probe_results', dst)
print('Saved to:', dst)


## Next

After this smoke test succeeds, sweep multiple joints, axes, perturbation magnitudes and frames. The first interpretable research result is the distribution of locality/leakage across poses and body regions, not a single wrist number.
